<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Fundamentos de automatización de procesos con Python**

## *De una tarea manual a un proceso automatizable*

Este laboratorio inicia **Data Gathery — Automatización Inteligente con Python e IA**.

Seguiremos durante todo el curso el patrón:

> **observar → medir → detectar inconsistencia → diseñar reglas → automatizar → validar → interpretar**

### Objetivos
- Identificar tareas automatizables.
- Diferenciar automatización tradicional e inteligente.
- Medir el coste de tareas repetitivas.
- Priorizar oportunidades.
- Traducir reglas humanas a Python.
- Introducir trazabilidad, revisión humana y riesgo.
- Preparar conceptualmente el Organizador Inteligente de Documentos.

## 1. Preparación

In [ ]:
from pathlib import Path
from collections import Counter
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
print("Entorno preparado.")

## 2. ¿Qué tareas son automatizables?

Una oficina puede repetir diariamente procesos como recibir archivos, clasificarlos, renombrarlos, generar informes o responder solicitudes.

Antes de programar debemos separar **reglas deterministas** de tareas que requieren **interpretación**.

In [ ]:
tareas = pd.DataFrame({
    "tarea": [
        "Renombrar facturas", "Mover PDFs por extensión",
        "Generar informe mensual", "Responder consulta ambigua",
        "Detectar intención de un correo", "Copiar archivos de backup",
        "Calcular totales de Excel", "Resumir contrato",
        "Enviar recordatorio periódico", "Priorizar incidencia"
    ],
    "repeticiones_mes": [250,400,12,80,300,30,120,40,60,150],
    "minutos_manual": [1,.5,20,4,2,3,5,15,2,3],
    "reglas_claras": [True,True,True,False,False,True,True,False,True,False],
    "requiere_interpretacion": [False,False,False,True,True,False,False,True,False,True]
})
tareas

### Primero medimos el problema

In [ ]:
tareas["minutos_mes"] = tareas["repeticiones_mes"] * tareas["minutos_manual"]
tareas["horas_mes"] = tareas["minutos_mes"] / 60
tareas.sort_values("horas_mes", ascending=False)

### Visualización de la carga manual

In [ ]:
orden = tareas.sort_values("horas_mes")
plt.figure(figsize=(10,6))
plt.barh(orden["tarea"], orden["horas_mes"])
plt.xlabel("Horas manuales / mes")
plt.title("Carga manual mensual por tarea")
plt.tight_layout()
plt.show()

### Interpretación

Una operación breve puede consumir muchas horas si se repite cientos de veces. La frecuencia es una variable central para priorizar automatizaciones.

## 3. Priorización de oportunidades

In [ ]:
tareas["puntuacion"] = (
    tareas["horas_mes"]
    * tareas["reglas_claras"].map({True:1.0, False:.35})
    * tareas["requiere_interpretacion"].map({True:.6, False:1.0})
)
tareas[["tarea","horas_mes","puntuacion"]].sort_values("puntuacion", ascending=False)

La puntuación anterior es **didáctica**, no una fórmula universal. Sirve para introducir la combinación de volumen, coste y dificultad.

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(tareas["repeticiones_mes"], tareas["minutos_manual"], s=100)
for _, f in tareas.iterrows():
    plt.annotate(f["tarea"], (f["repeticiones_mes"], f["minutos_manual"]), fontsize=8)
plt.xlabel("Repeticiones / mes")
plt.ylabel("Minutos / ejecución")
plt.title("Mapa de oportunidades de automatización")
plt.tight_layout()
plt.show()

## 4. Automatización tradicional vs inteligente

**Tradicional:** reglas claras, entradas estructuradas y comportamiento determinista.

**Inteligente:** interpretación de texto, documentos o situaciones ambiguas.

Una arquitectura profesional puede combinar ambas y derivar los casos inciertos a revisión humana.

In [ ]:
def tipo_automatizacion(reglas_claras, requiere_interpretacion):
    if reglas_claras and not requiere_interpretacion:
        return "Tradicional"
    if requiere_interpretacion:
        return "Potencialmente inteligente"
    return "Revisar"

tareas["tipo_propuesto"] = tareas.apply(
    lambda f: tipo_automatizacion(f["reglas_claras"], f["requiere_interpretacion"]),
    axis=1
)
tareas[["tarea","tipo_propuesto"]]

In [ ]:
conteo = tareas["tipo_propuesto"].value_counts()
plt.figure(figsize=(8,5))
plt.bar(conteo.index, conteo.values)
plt.ylabel("Número de tareas")
plt.title("Tipo de automatización propuesta")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 5. Caso: documentos desordenados

Primero veremos la inconsistencia. **No corregimos antes de observarla.**

In [ ]:
documentos = [
    "factura_2026_001.pdf", "clientes_agosto.xlsx", "foto_evento.jpg",
    "factura_2026_002.pdf", "informe_ventas.xlsx", "contrato_cliente.pdf",
    "logo_empresa.png", "notas_reunion.txt", "factura_2026_003.PDF",
    "datos.csv", "README", "presupuesto final.xlsx", "documento.tmp"
]
for documento in documentos:
    print(documento)

### Detectar extensiones

In [ ]:
extensiones = [Path(x).suffix for x in documentos]
Counter(extensiones)

Observamos `.pdf`/`.PDF`, un archivo sin extensión, `.tmp` y nombres con espacios. Ahora sí podemos visualizar el problema.

In [ ]:
conteo_ext = pd.Series(extensiones).value_counts()
plt.figure(figsize=(9,5))
plt.bar(conteo_ext.index.astype(str), conteo_ext.values)
plt.xlabel("Extensión")
plt.ylabel("Archivos")
plt.title("Extensiones antes de normalizar")
plt.tight_layout()
plt.show()

## 6. Corregir: normalización

In [ ]:
def extension_normalizada(nombre):
    ext = Path(nombre).suffix.lower()
    return ext if ext else "sin_extension"

normalizadas = [extension_normalizada(x) for x in documentos]
Counter(normalizadas)

In [ ]:
comparacion = pd.DataFrame({
    "archivo": documentos,
    "extension_original": extensiones,
    "extension_normalizada": normalizadas
})
comparacion

### Interpretación

El patrón es fundamental:

**inspeccionar → detectar inconsistencia → normalizar → aplicar reglas**.

## 7. Convertir reglas humanas en código

In [ ]:
REGLAS = {
    ".pdf":"PDF", ".xlsx":"Excel", ".csv":"Datos",
    ".jpg":"Imagenes", ".jpeg":"Imagenes", ".png":"Imagenes", ".txt":"Texto"
}

def clasificar_archivo(nombre):
    ext = Path(nombre).suffix.lower()
    if not ext:
        return "Sin_extension"
    return REGLAS.get(ext, "Otros")

clasificacion = pd.DataFrame({
    "archivo": documentos,
    "categoria": [clasificar_archivo(x) for x in documentos]
})
clasificacion

In [ ]:
categorias = clasificacion["categoria"].value_counts()
plt.figure(figsize=(9,5))
plt.bar(categorias.index, categorias.values)
plt.ylabel("Documentos")
plt.title("Documentos por categoría")
plt.tight_layout()
plt.show()

## 8. Casos que las reglas no entienden

In [ ]:
casos_revision = clasificacion[
    clasificacion["categoria"].isin(["Otros","Sin_extension"])
]
casos_revision

Una automatización segura no oculta los casos desconocidos:

```text
caso conocido → automatizar
caso desconocido → registrar → revisión humana
```

Más adelante la IA podrá ayudar con algunos casos ambiguos.

## 9. Medir ahorro potencial

In [ ]:
N = 10_000
SEGUNDOS_MANUAL = 12
manual = N * SEGUNDOS_MANUAL

docs = [
    f"factura_{i:05d}.pdf" if i % 3 == 0
    else f"datos_{i:05d}.csv" if i % 3 == 1
    else f"imagen_{i:05d}.jpg"
    for i in range(N)
]

inicio = time.perf_counter()
resultado_auto = [clasificar_archivo(x) for x in docs]
automatico = time.perf_counter() - inicio

print(f"Manual estimado: {manual/3600:.2f} horas")
print(f"Automático medido: {automatico:.6f} segundos")

In [ ]:
ahorro = (manual - automatico) / manual * 100
print(f"Ahorro estimado del escenario: {ahorro:.4f}%")

La comparación es un escenario didáctico. En un proyecto real mediríamos el proceso manual con datos observados.

## 10. Riesgo: no todo lo automatizable debe automatizarse sin controles

In [ ]:
riesgos = pd.DataFrame({
    "proceso": [
        "Mover copias de informes", "Renombrar fotografías",
        "Enviar factura", "Borrar documentos",
        "Clasificar reclamaciones", "Autorizar pago"
    ],
    "impacto_error": [1,1,4,5,3,5],
    "facilidad_reversion": [5,5,3,1,4,1],
    "requiere_revision": [False,False,True,True,True,True]
})
riesgos

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(riesgos["facilidad_reversion"], riesgos["impacto_error"], s=120)
for _, f in riesgos.iterrows():
    plt.annotate(f["proceso"], (f["facilidad_reversion"], f["impacto_error"]), fontsize=8)
plt.xlabel("Facilidad de reversión (1–5)")
plt.ylabel("Impacto del error (1–5)")
plt.title("Mapa simple de riesgo")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

Impacto alto y reversión difícil implican controles más estrictos. La pregunta profesional es **cómo automatizar de forma segura**, no solo si técnicamente podemos hacerlo.

## 11. Diseñar el proceso antes del código

In [ ]:
proceso = pd.DataFrame({
    "orden":[1,2,3,4,5,6],
    "paso":[
        "Detectar archivo","Validar nombre","Identificar extensión",
        "Clasificar","Registrar operación","Mover o enviar a revisión"
    ],
    "entrada":["carpeta","archivo","nombre válido","extensión","resultado","clasificación"],
    "salida":["archivo","nombre validado","extensión","categoría","log","archivo procesado"]
})
proceso

## 12. Primera automatización con trazabilidad

In [ ]:
def procesar_documentos(documentos):
    registros = []
    for archivo in documentos:
        categoria = clasificar_archivo(archivo)
        estado = "revision" if categoria in {"Otros","Sin_extension"} else "automatico"
        registros.append({
            "archivo": archivo,
            "categoria": categoria,
            "estado": estado
        })
    return pd.DataFrame(registros)

resultado = procesar_documentos(documentos)
resultado

### Validar antes de confiar

In [ ]:
print("Entradas:", len(documentos))
print("Salidas:", len(resultado))
assert len(documentos) == len(resultado)
assert resultado["categoria"].notna().all()
assert resultado["estado"].notna().all()
print("Validación superada.")

In [ ]:
estado = resultado["estado"].value_counts()
plt.figure(figsize=(7,5))
plt.bar(estado.index, estado.values)
plt.ylabel("Documentos")
plt.title("Automático frente a revisión")
plt.tight_layout()
plt.show()

## 13. Nueva inconsistencia: extensiones especiales

In [ ]:
problematicos = [
    "factura.pdf","FACTURA.PDF","informe.xlsx","archivo",
    ".gitignore","datos.CSV","foto.JPEG","backup.tar.gz"
]

pd.DataFrame({
    "archivo": problematicos,
    "categoria": [clasificar_archivo(x) for x in problematicos]
})

Antes de cambiar el clasificador inspeccionamos qué entiende realmente `pathlib`.

In [ ]:
for archivo in problematicos:
    ruta = Path(archivo)
    print(f"{archivo:20s} suffix={ruta.suffix!r:8s} suffixes={ruta.suffixes}")

Esto anticipa el laboratorio específico de `pathlib`, `os`, `shutil` y `glob`: los casos reales suelen ser más complejos que el ejemplo perfecto.

## 14. Arquitectura híbrida

```text
archivo
   ↓
¿regla determinista suficiente?
   ├── sí → Python
   └── no → componente inteligente
                ↓
          confianza suficiente?
           ├── sí → continuar
           └── no → revisión humana
```

La IA no sustituye necesariamente las reglas; puede ocuparse solo de la parte ambigua.

## 15. KPIs de automatización

In [ ]:
metricas = {
    "documentos_totales": len(resultado),
    "automaticos": int((resultado["estado"] == "automatico").sum()),
    "revision": int((resultado["estado"] == "revision").sum())
}
metricas["tasa_automatizacion"] = metricas["automaticos"] / metricas["documentos_totales"]

pd.DataFrame({
    "KPI":["Documentos","Automáticos","Revisión","Tasa automatización"],
    "valor":[
        metricas["documentos_totales"], metricas["automaticos"],
        metricas["revision"], f'{metricas["tasa_automatizacion"]:.1%}'
    ]
})

Más adelante ampliaremos esta observabilidad con **logging, excepciones, tiempos, backups y notificaciones**.

## 16. Mini-práctica guiada

Una oficina realiza mensualmente:

| Tarea | Frecuencia | Tiempo | Regla |
|---|---:|---:|---|
| Clasificar documentos | 600 | 20 s | extensión |
| Resumir correos | 300 | 3 min | contenido |
| Crear backup | 30 | 5 min | copiar carpeta |
| Preparar Excel | 20 | 30 min | reglas conocidas |
| Responder reclamaciones | 100 | 10 min | lenguaje natural |

Debes:

1. homogeneizar unidades;
2. calcular horas mensuales;
3. ordenar oportunidades;
4. clasificar tradicional/inteligente;
5. crear al menos dos gráficos;
6. identificar riesgos;
7. elegir las dos primeras automatizaciones que desarrollarías y justificarlo.

## 17. Ejercicios finales

1. **Inventario:** identifica diez tareas repetitivas.
2. **Coste:** añade frecuencia y tiempo medio.
3. **Priorización:** calcula horas mensuales.
4. **Reglas:** marca las tareas deterministas.
5. **IA:** identifica las que requieren interpretación.
6. **Visualización:** representa frecuencia frente a duración.
7. **Clasificador:** soporta al menos seis extensiones.
8. **Inconsistencia:** introduce extensiones en mayúsculas y observa el fallo antes de corregir.
9. **Desconocidos:** añade archivos sin extensión.
10. **Revisión humana:** crea un estado `revision`.
11. **Trazabilidad:** genera archivo, categoría y estado.
12. **Validación:** comprueba que cada entrada genera una salida.
13. **KPI:** calcula la tasa de automatización.
14. **Riesgo:** añade impacto y reversibilidad.
15. **Arquitectura híbrida:** combina reglas, IA y revisión humana.
16. **Reto:** diseña el flujo conceptual del Organizador Inteligente de Documentos del final del Módulo 1.

## 18. Preguntas de reflexión

1. ¿Qué hace que una tarea sea buena candidata?
2. ¿Por qué frecuencia y duración deben analizarse conjuntamente?
3. ¿Qué diferencia hay entre automatización tradicional e inteligente?
4. ¿Cuándo una regla es preferible a IA?
5. ¿Qué hacemos con casos desconocidos?
6. ¿Cuándo necesitamos revisión humana?
7. ¿Qué significa reversibilidad?
8. ¿Qué es trazabilidad?
9. ¿Qué deberíamos registrar?
10. ¿Por qué observar una inconsistencia antes de corregirla?
11. ¿Qué papel tendrá el logging?
12. ¿Qué parte del organizador documental puede resolverse sin IA?
13. ¿Qué parte podría beneficiarse de IA?
14. ¿Cómo medirías si la automatización mejora el proceso?

## 19. Cierre

Este laboratorio ha establecido el criterio para diseñar automatizaciones.

La progresión del Módulo 1 será:

```text
fundamentos
   ↓
estructuras de datos
   ↓
funciones y módulos
   ↓
pathlib / os / shutil / glob
   ↓
automatización de ficheros
   ↓
logging y errores
   ↓
Organizador Inteligente de Documentos
```